In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
import yt
import os
from dotenv import dotenv_values
import glob


In [ ]:
config = dotenv_values("config.sh")

Nx = int(config["Nx"])
Ny = int(config["Ny"])
SEED = config["SEED"]
MAX_GRID_SIZE = config["MAX_GRID_SIZE"]
FIXED_DT = config["FIXED_DT"]
VISC_COEF = config["VISC_COEF"]
CELL_DEPTH = config["CELL_DEPTH"]
NOISE_OFF_STEP = config["NOISE_OFF_STEP"]

# Create a dictionary of the variables we just extracted
params_table = {
    "Nx": Nx,
    "Ny": Ny,
    "SEED": SEED,
    "MAX_GRID_SIZE": MAX_GRID_SIZE,
    "FIXED_DT": FIXED_DT,
    "VISC_COEF": VISC_COEF,
    "CELL_DEPTH": CELL_DEPTH,
    "NOISE_OFF_STEP": NOISE_OFF_STEP
}

# Print them as a formatted table
print("=" * 40)
print(f"{'Parameter':<20} | {'Value'}")
print("=" * 40)

for key, value in params_table.items():
    print(f"{key:<20} | {value}")

print("=" * 40)

In [ ]:
raw_dir = f"snapshots_Nx{Nx}_Ny{Ny}_seed{SEED}_mgs{MAX_GRID_SIZE}_dt{FIXED_DT}_visc_coef{VISC_COEF}_noise_off_step{NOISE_OFF_STEP}_depth{CELL_DEPTH}"
run_dir = raw_dir.replace('.', 'p')
# plt_dir = os.path.join(run_dir, "plt")

plotfile_paths = sorted(glob.glob(os.path.join(run_dir, "plt0*")))

print(f"Found {len(plotfile_paths)} plotfiles in {run_dir}.")

ts = yt.DatasetSeries(plotfile_paths)

first_ds = ts[0]
print("\nAvailable variables in the plotfiles:")
for field in first_ds.field_list:
    print(f" - {field[1]}") 
    

In [ ]:
def frame_plot_helper(ax, data, title, cmap='RdBu', clim=None, use_log=False, fontsize=13, Lx=2*np.pi, Ly=2*np.pi):
    """Plot data with colorbar on top in separated axes"""
    
    plot_height, plot_bottom = 0.75, 0.05
    cbar_height, cbar_bottom = 0.04, 0.85
    plot_width, plot_left = 0.90, 0.075  
    
    pbbox = transforms.Bbox.from_bounds(plot_left, plot_bottom, plot_width, plot_height)
    cbbox = transforms.Bbox.from_bounds(plot_left, cbar_bottom, plot_width, cbar_height)
    
    to_axes_bbox = transforms.BboxTransformTo(ax.get_position())
    pbbox = pbbox.transformed(to_axes_bbox)
    cbbox = cbbox.transformed(to_axes_bbox)
    
    paxes = ax.figure.add_axes(pbbox)
    caxes = ax.figure.add_axes(cbbox)
    ax.axis('off')
    
    # Optional: If you have a set_pi_ticks function defined elsewhere, keep this.
    # set_pi_ticks(paxes, denom=2)
    
    vmin, vmax = data.min(), data.max()
    vlim = max(abs(vmin), abs(vmax))
    
    extent = [0, Lx, -Ly/2, Ly/2]  
    # print(extent)
    
    if clim is None:
        im = paxes.imshow(data, cmap=cmap, origin='lower', extent=extent, vmin=-vlim, vmax=vlim)
    else:
        im = paxes.imshow(data, cmap=cmap, origin='lower', vmin=clim[0], vmax=clim[1], extent=extent)

    if use_log:
        custom_cmap = plt.cm.Reds.copy()
        custom_cmap.set_under('white')
        
        im = paxes.imshow(np.abs(data), cmap=custom_cmap, origin='lower', extent=extent, 
                          aspect=1, norm=LogNorm(vmin=1e-1, vmax=vlim))
    else:
        if clim is None:
            im = paxes.imshow(data, cmap=cmap, origin='lower', extent=extent, vmin=-vlim, vmax=vlim)
        else:
            im = paxes.imshow(data, cmap=cmap, origin='lower', vmin=clim[0], vmax=clim[1], extent=extent)
    
    paxes.set_xlabel('x', fontsize=fontsize)
    paxes.set_ylabel('y', fontsize=fontsize)
    paxes.tick_params(length=0, width=0, labelsize=fontsize-2)
    
    caxes.text(0.5, 3.1, title, transform=caxes.transAxes, 
               ha='center', va='bottom', fontsize=fontsize)
    
    cbar = plt.colorbar(im, cax=caxes, orientation='horizontal',
                       ticks=ticker.MaxNLocator(nbins=5))
    cbar.outline.set_visible(False)
    caxes.xaxis.set_ticks_position('top')
    cbar.ax.tick_params(labelsize=fontsize-2)
    
    return paxes, caxes
    
yt.funcs.mylog.setLevel(30)

nrows = 1
ncols = 4

for i in range(36):
    ds = ts[i]
    sim_time = ds.current_time.v  # .v gets the raw float value without yt units
    
    cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
    
    # Now we extract the variables from the covering grid
    # We use [:, :, 0] to slice off the dummy z-dimension
    magvort = np.array(cg['magvort'][:, :, 0].v)
    avg_velx = np.array(cg['averaged_velx'][:, :, 0].v)
    avg_vely = np.array(cg['averaged_vely'][:, :, 0].v)
    
    fig_width = 5 * ncols
    fig_height = 5 * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width, fig_height), dpi=200)
    
    fig.suptitle(f't = {sim_time:.5e}', fontsize=15, y=0.94)
    
    if nrows * ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
        
    clim = None
    
    # Note: We transpose (.T) because imshow expects shape (Ny, Nx)
    frame_plot_helper(axes[0], magvort.T, 'Vorticity', cmap='RdBu', clim=clim)
    frame_plot_helper(axes[1], magvort.T, r'$\log$|Vorticity|', use_log=True)
    frame_plot_helper(axes[2], avg_velx.T, r'$\langle U_x \rangle$', cmap='RdBu', clim=clim)
    frame_plot_helper(axes[3], avg_vely.T, r'$\langle U_y \rangle$', cmap='RdBu', clim=clim)
    
    plt.show()